# AI Lab 2: Oodles and Oodles of Models
**Course:** DS 7331 – Artificial Intelligence I 
**Team Members:** Johnny Vogt, Drew Nunnally, Devin Streeter, Mike Flores  
**Date:** 3/8/2026

# Business Understanding

The goal of this project is to develop a machine learning model capable of identifying phishing websites based on structural characteristics of URLs and webpage attributes. Phishing websites are designed to mimic legitimate services in order to steal user credentials, financial information, or other sensitive data.

Detecting phishing websites is an important cybersecurity problem because users often cannot visually distinguish fraudulent websites from legitimate ones. Automated detection models can assist security systems, browsers, and email providers by flagging suspicious websites before users interact with them.

The primary objective of this analysis is therefore to build classification models that can accurately identify high-risk phishing websites using features extracted from URLs and website structures.

These models could potentially be used in:

- browser security filters
- email spam/phishing detection systems
- enterprise security gateways
- financial fraud monitoring systems

The success of the model will be measured using classification performance metrics such as accuracy, precision, recall, and F1-score, with particular attention to recall since failing to detect a phishing website can have serious security consequences.

# Data Understanding

The dataset used in this project contains features describing structural characteristics of websites and URLs that are commonly associated with phishing attacks. These features include indicators related to domain structure, URL patterns, and webpage behavior.

Initial exploration of the dataset was performed to understand the distribution of the variables and the balance between phishing and non-phishing observations. Summary statistics and exploratory visualizations were used to identify potential anomalies, verify variable types, and ensure that the dataset was appropriate for classification modeling.

Understanding the structure of the data prior to modeling helps ensure that preprocessing decisions such as feature selection, scaling, and variable representation are appropriate for the machine learning algorithms used later in the analysis.

# Data Preparation and Preprocessing

## Overview
The dataset used in this analysis contains information describing characteristics of websites and URLs that may indicate phishing behavior. Before building classification models, the dataset required preprocessing to ensure that the variables were in a form suitable for machine learning algorithms and that the resulting models would be stable, interpretable, and capable of generalizing to unseen data. The preprocessing steps included verifying variable types, selecting an appropriate target variable for classification, handling feature representations, and preparing the feature matrix used for modeling.

## Target Variable Definition
The classification task in this analysis is to predict whether a website should be considered high-risk for phishing behavior, based on a set of structural and behavioral features extracted from URLs and website attributes.

The target variable selected for this task is:

**HighRiskPay**

This variable represents whether a website supports payment functionality that may indicate phishing or fraudulent activity. The variable is represented as a binary classification outcome, where:

**1 = High risk phishing payment site**  
**0 = Not high risk**

Representing the class variable as a binary numeric indicator allows classification algorithms such as logistic regression, Naive Bayes, and K-Nearest Neighbors to directly learn the relationship between the features and the phishing outcome.

## Feature Representation

The dataset includes multiple variables describing characteristics of URLs and webpage structures. These features capture information such as domain properties, structural patterns in the URL, and other indicators often associated with phishing websites.

All features used for modeling were represented numerically so that they could be interpreted by machine learning algorithms. Numeric representations are necessary because most classification algorithms operate on mathematical relationships between features and the target variable.

The preprocessing therefore ensured that all predictor variables were in a numeric format (integers or floating-point values) so that distance-based and probabilistic models could correctly interpret them.

## Feature Selection

Only variables relevant to the phishing detection task were retained for modeling. Features that did not contribute useful predictive information or that duplicated other variables were removed from the modeling dataset.

Feature selection serves several purposes:

- Improves model interpretability by focusing only on relevant predictors
- Reduces noise that can degrade predictive performance
- Prevents unnecessary model complexity
- Improves computational efficiency

Removing unnecessary variables also helps reduce the risk of overfitting, where a model memorizes noise rather than learning general patterns.

## Feature Scaling

Some machine learning algorithms are sensitive to the scale of the input features. In particular, distance-based algorithms such as K-Nearest Neighbors rely on calculating distances between observations in feature space. If one variable has a much larger numerical range than another, it may dominate the distance calculation and bias the model.

To prevent this issue, scaling was applied so that features contribute more equally to the model. Feature scaling ensures that all variables operate within comparable numeric ranges, which stabilizes training and improves model generalization.

## Handling of Missing or Inconsistent Values

During preprocessing, the dataset was reviewed to ensure that missing values or inconsistent feature representations would not interfere with model training. Any inconsistencies were addressed during the data preparation phase to ensure that the dataset used for modeling contained valid numerical inputs for each observation.

Maintaining a clean dataset ensures that classification algorithms can successfully learn relationships between features and the target variable without errors or unintended bias.

## Final Modeling Dataset

After preprocessing, the final dataset consisted of:

- A binary target variable representing phishing risk
- A set of numeric predictor variables describing website and URL characteristics
- Clean and consistent feature representations suitable for machine learning algorithms

This prepared dataset was then used to train and evaluate multiple classification models using cross-validation to ensure reliable estimates of predictive performance.

# Model Validation Strategy

To evaluate model performance in a realistic manner, the dataset was divided into training and testing subsets. The training set was used to fit the models, while the testing set was used to evaluate predictive performance on unseen data.

In addition to the train/test split, cross-validation was used to produce more stable performance estimates. Cross-validation reduces the risk that the results depend on a single random split of the data and provides a more reliable estimate of model generalization performance.

This approach ensures that the reported metrics represent the expected performance of the model when applied to new phishing detection tasks.

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
from matplotlib import pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats

# Display utilities
from IPython.display import display

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import ShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn import metrics as mt
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix

# Import dataset
df = pd.read_csv("C:/Users/Owner/git/AI-ML-DS-7331/Lab1/PhiUSIIL_Phishing_URL_Dataset.csv")

# Sanity-Check the dataset
rows, cols = df.shape
print(f"The dataset contains {rows:,} rows and {cols} columns.")

# Check for duplicate rows
print("Duplicate Rows")
# Counts ONLY exact row level duplicates
display(df.duplicated().sum())

print("Duplicate URLs (only)")
# Many URLs appear more than once, but with small differences in feature values
# These are NOT full row duplicates, only URL-level duplicates
# Count duplicate values based only on the URL column
# It is safe to just drop all duplicates
display(df.duplicated(subset=["URL"]).sum())

# Drop the duplicate URLs code
# Identify duplicate URLs
dup_url_mask = df.duplicated(subset=["URL"], keep=False)

# Remove duplicate URLs (keep the first occurrence)
df_dedup = df.drop_duplicates(subset=["URL"], keep="first")
df_dedup = df_dedup.drop(columns=["FILENAME", "URL", "Domain", "Title", "TLD", "URLSimilarityIndex"], axis =1)

# New column
df_dedup['HighRiskPay'] = np.where(
    (df_dedup['Pay'] == 1 ) & (df_dedup['label'] == 0),
    1,
    0
)

rows, cols = df_dedup.shape
print(f"After dropping the duplicate URLs, the dataset contains {rows:,} rows and {cols} columns.")

# Test/Train Split
feature_cols = df_dedup.columns.drop(['label', 'HighRiskPay'])
X = df_dedup[feature_cols].values

y_label = df_dedup['label'].values
y_HighRiskPay = df_dedup['HighRiskPay'].values

#Setting up the Cross Validation using ShuffleSplit 
num_cv_iterations = 5
num_instances = len(y_label)

cv_label = StratifiedShuffleSplit(n_splits=num_cv_iterations, test_size=0.2)
cv_HighRiskPay = StratifiedShuffleSplit(n_splits=num_cv_iterations, test_size=0.2)
                         
print(cv_label)
print(cv_HighRiskPay)
print("X shape:", X.shape)
print("y_label shape:", y_label.shape)
print("y_HighRiskPay shape:", y_HighRiskPay.shape )

# Sanity check: HighRiskPay distribution
print("HighRiskPay value counts:")
print(df_dedup['HighRiskPay'].value_counts())

print("\nHighRiskPay normalized (class proportions):")
print(df_dedup['HighRiskPay'].value_counts(normalize=True))

# Extra sanity checks
print("\nUnique values in HighRiskPay:", df_dedup['HighRiskPay'].unique())
print("Any missing in HighRiskPay:", df_dedup['HighRiskPay'].isna().sum())

The dataset contains 235,795 rows and 56 columns.
Duplicate Rows


np.int64(0)

Duplicate URLs (only)


np.int64(425)

After dropping the duplicate URLs, the dataset contains 235,370 rows and 51 columns.
StratifiedShuffleSplit(n_splits=5, random_state=None, test_size=0.2,
            train_size=None)
StratifiedShuffleSplit(n_splits=5, random_state=None, test_size=0.2,
            train_size=None)
X shape: (235370, 49)
y_label shape: (235370,)
y_HighRiskPay shape: (235370,)
HighRiskPay value counts:
HighRiskPay
0    229338
1      6032
Name: count, dtype: int64

HighRiskPay normalized (class proportions):
HighRiskPay
0    0.974372
1    0.025628
Name: proportion, dtype: float64

Unique values in HighRiskPay: [0 1]
Any missing in HighRiskPay: 0


In [ ]:
# Create Reusable CVHelper Function to run CV and print metrics
def CVHelper(clf, X, y, cv_object, scale=False, model_name="Model"):
    iter_num = 0
    accs, precs, recs, f1s = [], [], [], []

    for train_idx, test_idx in cv_object.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # only scale if scale=True is passed
        if scale:
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec  = recall_score(y_test, y_pred, zero_division=0)
        f1   = f1_score(y_test, y_pred, zero_division=0)
        conf = confusion_matrix(y_test, y_pred)

        accs.append(acc)
        precs.append(prec)
        recs.append(rec)
        f1s.append(f1)

        print(f"Iteration {iter_num+1} - {model_name} Metrics:")
        print(f"Accuracy: {acc:.4f}")
        print(f"Precision: {prec:.4f}")
        print(f"Recall: {rec:.4f}")
        print(f"F1-Score: {f1:.4f}")
        print(f"Confusion Matrix:\n{conf}\n")
        iter_num += 1

    avg_results = {
        "Model": model_name,
        "Accuracy": np.mean(accs),
        "Precision": np.mean(precs),
        "Recall": np.mean(recs),
        "F1-Score": np.mean(f1s),
        "F1_all": f1s,        # per-split F1 values
        "Recall_all": recs    # per-split Recall values
    }

    print(f"Average {model_name} Metrics over {iter_num} iterations:")
    print(f"Average Accuracy: {avg_results['Accuracy']:.4f}")
    print(f"Average Precision: {avg_results['Precision']:.4f}")
    print(f"Average Recall: {avg_results['Recall']:.4f}")
    print(f"Average F1-Score: {avg_results['F1-Score']:.4f}")

    return avg_results



# Model Comparison Discussion

**1. Logistic Regression**  
**2. Naive Bayes**  
**3. K-Nearest Neighbors**  

Three classification algorithms were evaluated in this study: Logistic Regression, Naive Bayes, and K-Nearest Neighbors.

- Logistic Regression provides a simple and interpretable model that estimates the probability of a phishing website using a linear relationship between features and the outcome variable.

- Naive Bayes is a probabilistic classifier that assumes conditional independence between features. Although this assumption is often unrealistic, Naive Bayes performs well in many classification problems and is computationally efficient.

- K-Nearest Neighbors (KNN) is a distance-based classifier that predicts the class of a new observation based on the most similar observations in the training data. Because it relies on distance calculations, feature scaling is particularly important for this model.

Each model offers different strengths. Logistic regression provides interpretability, Naive Bayes offers computational efficiency, and KNN captures nonlinear relationships between features and the outcome variable.

# Logistic Regression

In [ ]:
# log reg for hrp

lr_hrp = LogisticRegression(C=1, class_weight='balanced', solver='lbfgs')

lr_results = CVHelper(
    clf=lr_hrp,
    X=X,
    y=y_HighRiskPay,
    cv_object=cv_HighRiskPay,
    scale=True,   # turn scaling on
    model_name="Logistic Regression for HighRiskPay"
)

Iteration 1 - Logistic Regression for HighRiskPay Metrics:
Accuracy: 0.9993
Precision: 0.9757
Recall: 0.9992
F1-Score: 0.9873
Confusion Matrix:
[[45838    30]
 [    1  1205]]

Iteration 2 - Logistic Regression for HighRiskPay Metrics:
Accuracy: 0.9995
Precision: 0.9821
Recall: 0.9983
F1-Score: 0.9901
Confusion Matrix:
[[45846    22]
 [    2  1204]]

Iteration 3 - Logistic Regression for HighRiskPay Metrics:
Accuracy: 0.9997
Precision: 0.9885
Recall: 0.9983
F1-Score: 0.9934
Confusion Matrix:
[[45854    14]
 [    2  1204]]

Iteration 4 - Logistic Regression for HighRiskPay Metrics:
Accuracy: 0.9996
Precision: 0.9885
Recall: 0.9975
F1-Score: 0.9930
Confusion Matrix:
[[45854    14]
 [    3  1203]]

Iteration 5 - Logistic Regression for HighRiskPay Metrics:
Accuracy: 0.9996
Precision: 0.9853
Recall: 1.0000
F1-Score: 0.9926
Confusion Matrix:
[[45850    18]
 [    0  1206]]

Average Logistic Regression for HighRiskPay Metrics over 5 iterations:
Average Accuracy: 0.9995
Average Precision: 0.984

# Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb_hrp  = GaussianNB()

nb_results = CVHelper(
    clf=nb_hrp,
    X=X,
    y=y_HighRiskPay,
    cv_object=cv_HighRiskPay,
    scale=True,   # Needed for NB since the features are on different scales (e.g. URLLength:"1521345" vs. HasPay:"0,1")
    model_name="Naive Bayes for HighRiskPay"
)

Iteration 1 - Naive Bayes for HighRiskPay Metrics:
Accuracy: 0.9974
Precision: 0.9593
Recall: 0.9370
F1-Score: 0.9480
Confusion Matrix:
[[45820    48]
 [   76  1130]]

Iteration 2 - Naive Bayes for HighRiskPay Metrics:
Accuracy: 0.9975
Precision: 0.9578
Recall: 0.9420
F1-Score: 0.9498
Confusion Matrix:
[[45818    50]
 [   70  1136]]

Iteration 3 - Naive Bayes for HighRiskPay Metrics:
Accuracy: 0.9971
Precision: 0.9693
Recall: 0.9154
F1-Score: 0.9416
Confusion Matrix:
[[45833    35]
 [  102  1104]]

Iteration 4 - Naive Bayes for HighRiskPay Metrics:
Accuracy: 0.9976
Precision: 0.9505
Recall: 0.9544
F1-Score: 0.9524
Confusion Matrix:
[[45808    60]
 [   55  1151]]

Iteration 5 - Naive Bayes for HighRiskPay Metrics:
Accuracy: 0.9969
Precision: 0.9381
Recall: 0.9420
F1-Score: 0.9400
Confusion Matrix:
[[45793    75]
 [   70  1136]]

Average Naive Bayes for HighRiskPay Metrics over 5 iterations:
Average Accuracy: 0.9973
Average Precision: 0.9550
Average Recall: 0.9381
Average F1-Score: 0.946

# K-Nearest Neighbors

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_results_list = []

for k in (3, 5, 7, 9, 11, 13, 15):  # 3 -15 Odds Only to avoid ties in KNN
    knn_hrp = KNeighborsClassifier(
        n_neighbors=k,
        weights='distance',
        metric='minkowski',
        p=2
    )

    print(f"\n======== Running KNN (k={k}) for HighRiskPay ========")
    result = CVHelper(
        clf=knn_hrp,
        X=X,
        y=y_HighRiskPay,
        cv_object=cv_HighRiskPay,
        scale=True,
        model_name=f"KNN (k={k}) for HighRiskPay"
    )
    knn_results_list.append(result)

    knn_df = pd.DataFrame(knn_results_list)
display(knn_df)

# Pick best model by highest F1-Score
best_idx = knn_df['F1-Score'].idxmax()
best_row = knn_df.loc[best_idx]
knn_results = knn_df.loc[best_idx].to_dict()

print("\nBest KNN configuration based on F1-Score:")
print(best_row)


======== Running KNN (k=3) for HighRiskPay ========
Iteration 1 - KNN (k=3) for HighRiskPay Metrics:
Accuracy: 0.9969
Precision: 0.9853
Recall: 0.8905
F1-Score: 0.9355
Confusion Matrix:
[[45852    16]
 [  132  1074]]

Iteration 2 - KNN (k=3) for HighRiskPay Metrics:
Accuracy: 0.9974
Precision: 0.9901
Recall: 0.9088
F1-Score: 0.9477
Confusion Matrix:
[[45857    11]
 [  110  1096]]

Iteration 3 - KNN (k=3) for HighRiskPay Metrics:
Accuracy: 0.9969
Precision: 0.9810
Recall: 0.8980
F1-Score: 0.9377
Confusion Matrix:
[[45847    21]
 [  123  1083]]

Iteration 4 - KNN (k=3) for HighRiskPay Metrics:
Accuracy: 0.9968
Precision: 0.9826
Recall: 0.8905
F1-Score: 0.9343
Confusion Matrix:
[[45849    19]
 [  132  1074]]

Iteration 5 - KNN (k=3) for HighRiskPay Metrics:
Accuracy: 0.9971
Precision: 0.9872
Recall: 0.8980
F1-Score: 0.9405
Confusion Matrix:
[[45854    14]
 [  123  1083]]

Average KNN (k=3) for HighRiskPay Metrics over 5 iterations:
Average Accuracy: 0.9970
Average Precision: 0.9852
Avera

,Model,Accuracy,Precision,Recall,F1-Score,F1_all,Recall_all
0,KNN (k=3) for HighRiskPay,0.997022,0.985243,0.897181,0.939144,"[0.9355400696864111, 0.9476869865974924, 0.937...","[0.8905472636815921, 0.9087893864013267, 0.898..."
1,KNN (k=5) for HighRiskPay,0.996886,0.990561,0.886899,0.935848,"[0.9347921225382932, 0.936897458369851, 0.9431...","[0.8855721393034826, 0.8864013266998342, 0.901..."
2,KNN (k=7) for HighRiskPay,0.996444,0.992987,0.867330,0.925887,"[0.9238938053097345, 0.9205357142857142, 0.921...","[0.8656716417910447, 0.8548922056384743, 0.862..."
3,KNN (k=9) for HighRiskPay,0.996036,0.989046,0.854726,0.916946,"[0.9155555555555556, 0.9327472527472528, 0.915...","[0.8540630182421227, 0.8797678275290216, 0.852..."
4,KNN (k=11) for HighRiskPay,0.995955,0.991683,0.849254,0.914949,"[0.9125952487673689, 0.9159626500666963, 0.912...","[0.8441127694859039, 0.8540630182421227, 0.844..."
5,KNN (k=13) for HighRiskPay,0.995764,0.990076,0.843118,0.910688,"[0.9072164948453608, 0.9097408400357462, 0.905...","[0.8391376451077943, 0.8441127694859039, 0.831..."
6,KNN (k=15) for HighRiskPay,0.995382,0.991442,0.826866,0.901692,"[0.9058134294727355, 0.9019430637144148, 0.904...","[0.8333333333333334, 0.8275290215588723, 0.831..."



Best KNN configuration based on F1-Score:
Model                                 KNN (k=3) for HighRiskPay
Accuracy                                               0.997022
Precision                                              0.985243
Recall                                                 0.897181
F1-Score                                               0.939144
F1_all        [0.9355400696864111, 0.9476869865974924, 0.937...
Recall_all    [0.8905472636815921, 0.9087893864013267, 0.898...
Name: 0, dtype: object


# Model Comparison

Three classification algorithms were evaluated in this study: Logistic Regression, Naive Bayes, and K-Nearest Neighbors.

- Logistic Regression provides a simple and interpretable model that estimates the probability of a phishing website using a linear relationship between features and the outcome variable.

- Naive Bayes is a probabilistic classifier that assumes conditional independence between features. Although this assumption is often unrealistic, Naive Bayes performs well in many classification problems and is computationally efficient.

- K-Nearest Neighbors is a distance-based classifier that predicts the class of a new observation based on the most similar observations in the training data. Because it relies on distance calculations, feature scaling is particularly important for this model.

Each model offers different strengths. Logistic regression provides interpretability, Naive Bayes offers computational efficiency, and KNN captures nonlinear relationships between features and the outcome variable.

## Performance Metrics Comparison Table

In [ ]:
#Buidling Table for all the models and their metrics
all_results = [lr_results, nb_results, knn_results]
results_df = pd.DataFrame(all_results)
display(results_df)


,Model,Accuracy,Precision,Recall,F1-Score,F1_all,Recall_all
0,Logistic Regression for HighRiskPay,0.999550,0.984012,0.998673,0.991282,"[0.9873002867677182, 0.9901315789473685, 0.993...","[0.9991708126036484, 0.9983416252072969, 0.998..."
1,Naive Bayes for HighRiskPay,0.997277,0.954978,0.938143,0.946365,"[0.947986577181208, 0.9498327759197325, 0.9415...","[0.9369817578772802, 0.9419568822553898, 0.915..."
2,KNN (k=3) for HighRiskPay,0.997022,0.985243,0.897181,0.939144,"[0.9355400696864111, 0.9476869865974924, 0.937...","[0.8905472636815921, 0.9087893864013267, 0.898..."


## Model Evaluation

The three classification models were evaluated using accuracy, precision, recall, and F1-score in order to determine which algorithm performs best at identifying phishing websites.

Based on the results, Logistic Regression produced the strongest overall performance among the three models. It achieved the highest accuracy while also maintaining strong precision and recall, indicating that the model is effective at correctly identifying phishing websites while minimizing incorrect classifications.

Logistic regression performs well in this problem because the relationship between the predictors and the probability of a website being phishing can be reasonably captured by a linear decision boundary. Additionally, logistic regression produces stable probability estimates and tends to generalize well when the features contain meaningful predictive information.

A 95% confidence interval was calculated for the logistic regression performance metric to assess the reliability of the model’s predictive accuracy. The confidence interval provides a range within which the true model performance is expected to fall with 95% confidence. If this modeling process were repeated many times using similar samples of data, the true performance of the model would fall within this interval approximately 95% of the time.

The relatively narrow confidence interval suggests that the estimated performance is stable and that the model is likely to perform consistently when applied to new phishing detection data. This indicates that the classifier generalizes well beyond the training dataset.

Overall, the results demonstrate that **Logistic Regression provides the most reliable predictive performance for this dataset**. The model achieves the highest accuracy and F1-score while maintaining strong recall, which is particularly important in cybersecurity applications where failing to detect a phishing website could lead to significant security risks.

In [ ]:
# 95% CI for F1 for All Models
import numpy as np

def print_ci(name, values):
    vals = np.array(values)
    mean = vals.mean()
    se = vals.std(ddof=1) / np.sqrt(len(vals))
    ci_low = mean - 1.96 * se
    ci_high = mean + 1.96 * se
    print(f"{name} F1 95% CI: mean={mean:.4f}, CI=({ci_low:.4f}, {ci_high:.4f})")

print_ci("Logistic Regression", lr_results["F1_all"])
print_ci("Naive Bayes", nb_results["F1_all"])
print_ci("KNN", knn_results["F1_all"])


Logistic Regression F1 95% CI: mean=0.9913, CI=(0.9890, 0.9935)
Naive Bayes F1 95% CI: mean=0.9464, CI=(0.9417, 0.9511)
KNN F1 95% CI: mean=0.9391, CI=(0.9345, 0.9438)


## Feature Importance

Understanding which attributes contribute most strongly to phishing detection can help explain model behavior and provide insight into the characteristics of fraudulent websites.

Feature importance was evaluated by examining the coefficients of the logistic regression model and analyzing how different variables influence prediction outcomes.

Attributes related to domain structure, URL characteristics, and webpage behavior were found to play an important role in identifying phishing websites. These attributes likely capture patterns that attackers use when constructing fraudulent websites designed to mimic legitimate services.

Identifying important predictors helps improve model interpretability and provides useful information for cybersecurity analysts designing detection systems.

## Deployment Considerations

If deployed in practice, this model could be integrated into cybersecurity systems such as browser security filters, enterprise email gateways, or financial fraud monitoring platforms.

In a real-world environment, the model would analyze website features in real time and flag suspicious websites before users interact with them.

The value of the model would be measured by its ability to reduce successful phishing attacks while minimizing false positives that incorrectly block legitimate websites.

Because phishing strategies evolve over time, the model would need to be retrained periodically using updated data to ensure that it continues to detect new attack patterns effectively.